# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

In [1]:
from IPython.display import HTML, display
def set_output_wrapping():
    display(HTML('''
    <style>
    pre {
        white-space: pre-wrap;
        word-wrap: break-word;
        max-width: 100%;
        overflow-x: hidden;
    }
    .output_area {
        white-space: pre-wrap;
        word-wrap: break-word;
        max-width: 100%;
        overflow-x: hidden;
    }
    .output_text {
        white-space: pre-wrap;
        word-wrap: break-word;
    }
    div.output {
        white-space: pre-wrap;
        word-wrap: break-word;
        max-width: 100%;
    }
    span {
        white-space: pre-wrap;
        word-wrap: break-word;
    }
    </style>
    '''))
set_output_wrapping()

In [2]:
import getpass
import os


def set_api_key(key_name: str) -> None:
    """
    Securely set an environment variable if it doesn't already exist.
    Prompts the user for input using a password-style hidden input.
    
    Args:
        key_name (str): Name of the environment variable to set (e.g., "OPENAI_API_KEY")
    """
    if not os.environ.get(key_name):
        os.environ[key_name] = getpass.getpass(f"{key_name}: ")

# Example usage:
# set_api_key("OPENAI_API_KEY")
# set_api_key("LANGCHAIN_API_KEY")

# set_api_key("ANTHROPIC_API_KEY")

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jmichaeldean/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jmichaeldean/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [4]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
set_api_key("LANGCHAIN_API_KEY")

We'll also want to set a project name to make things easier for ourselves.

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [6]:
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
set_api_key("OPENAI_API_KEY")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [7]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [41]:
len(docs)

64

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [11]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '5660ec'. Skipping!
Property 'summary' already exists in node '1ebf13'. Skipping!
Property 'summary' already exists in node 'aefa3b'. Skipping!
Property 'summary' already exists in node 'f12de5'. Skipping!
Property 'summary' already exists in node 'ae9d75'. Skipping!
Property 'summary' already exists in node 'ce20be'. Skipping!
Property 'summary' already exists in node '1ae484'. Skipping!
Property 'summary' already exists in node '44ca68'. Skipping!
Property 'summary' already exists in node '50070a'. Skipping!
Property 'summary' already exists in node '492c72'. Skipping!
Property 'summary' already exists in node 'b774b9'. Skipping!
Property 'summary' already exists in node 'f32f20'. Skipping!
Property 'summary' already exists in node '3614ef'. Skipping!
Property 'summary' already exists in node '002e11'. Skipping!
Property 'summary' already exists in node 'ec7267'. Skipping!
Property 'summary' already exists in node '2c2c34'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'aefa3b'. Skipping!
Property 'summary_embedding' already exists in node '5660ec'. Skipping!
Property 'summary_embedding' already exists in node 'ce20be'. Skipping!
Property 'summary_embedding' already exists in node '1ebf13'. Skipping!
Property 'summary_embedding' already exists in node 'f12de5'. Skipping!
Property 'summary_embedding' already exists in node 'ae9d75'. Skipping!
Property 'summary_embedding' already exists in node '50070a'. Skipping!
Property 'summary_embedding' already exists in node '44ca68'. Skipping!
Property 'summary_embedding' already exists in node '492c72'. Skipping!
Property 'summary_embedding' already exists in node 'b774b9'. Skipping!
Property 'summary_embedding' already exists in node '1ae484'. Skipping!
Property 'summary_embedding' already exists in node 'ec7267'. Skipping!
Property 'summary_embedding' already exists in node '002e11'. Skipping!
Property 'summary_embedding' already exists in node '3614ef'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 712)

We can save and load our knowledge graphs as follows.

In [12]:
kg.save("usecase4_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase4_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 712)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [15]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,As a Knowledge Worker utilizing AI tools like ...,[Introduction ChatGPT launched in November 202...,"ChatGPT, launched in November 2022, has experi...",single_hop_specifc_query_synthesizer
1,When is June 2025?,[Table 1: ChatGPT daily message counts (millio...,The context reports data ending on the 26th of...,single_hop_specifc_query_synthesizer
2,What is SOC2 code 15?,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,Who is Brynjolfsson?,[Conclusion This paper studies the rapid growt...,The context does not provide specific informat...,single_hop_specifc_query_synthesizer
4,Considering the increase in ChatGPT message vo...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"Between June 2024 and June 2025, ChatGPT usage...",multi_hop_abstract_query_synthesizer
5,How does the global diffusion of new tech like...,[<1-hop>\n\nConclusion This paper studies the ...,The context indicates that ChatGPT's usage has...,multi_hop_abstract_query_synthesizer
6,How do AI applications like ChatGPT impact wor...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, launched in November 2022 and rapidly...",multi_hop_abstract_query_synthesizer
7,Based on the data indicating that 18 billion m...,[<1-hop>\n\nConclusion This paper studies the ...,"The context shows that by July 2025, ChatGPT u...",multi_hop_specific_query_synthesizer
8,Whre US is the chatgpt usage growng fast?,[<1-hop>\n\nConclusion This paper studies the ...,"According to the context, ChatGPT usage has gr...",multi_hop_specific_query_synthesizer
9,"Hwo Handa et al., 2025 and Handa et al. are th...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"Handa et al., 2025 and Handa et al. refer to t...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
# dataset = generator.generate_with_langchain_docs(docs, testset_size=10)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '912ab0'. Skipping!
Property 'summary' already exists in node 'b7fcc0'. Skipping!
Property 'summary' already exists in node 'a14219'. Skipping!
Property 'summary' already exists in node 'b97249'. Skipping!
Property 'summary' already exists in node '4f2fa3'. Skipping!
Property 'summary' already exists in node 'c16453'. Skipping!
Property 'summary' already exists in node '84173a'. Skipping!
Property 'summary' already exists in node '15782c'. Skipping!
Property 'summary' already exists in node 'e89518'. Skipping!
Property 'summary' already exists in node '0e7be8'. Skipping!
Property 'summary' already exists in node 'c8afda'. Skipping!
Property 'summary' already exists in node '301165'. Skipping!
Property 'summary' already exists in node '2efe48'. Skipping!
Property 'summary' already exists in node 'ca8016'. Skipping!
Property 'summary' already exists in node '9ce752'. Skipping!
Property 'summary' already exists in node '4e0a96'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '4f2fa3'. Skipping!
Property 'summary_embedding' already exists in node 'b7fcc0'. Skipping!
Property 'summary_embedding' already exists in node 'b97249'. Skipping!
Property 'summary_embedding' already exists in node 'e89518'. Skipping!
Property 'summary_embedding' already exists in node '912ab0'. Skipping!
Property 'summary_embedding' already exists in node '84173a'. Skipping!
Property 'summary_embedding' already exists in node 'ca8016'. Skipping!
Property 'summary_embedding' already exists in node 'c16453'. Skipping!
Property 'summary_embedding' already exists in node '0e7be8'. Skipping!
Property 'summary_embedding' already exists in node '15782c'. Skipping!
Property 'summary_embedding' already exists in node 'a14219'. Skipping!
Property 'summary_embedding' already exists in node 'c8afda'. Skipping!
Property 'summary_embedding' already exists in node '2efe48'. Skipping!
Property 'summary_embedding' already exists in node '301165'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [17]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What role does Ling play in the context of AI ...,[Introduction ChatGPT launched in November 202...,The provided context does not specify the role...,single_hop_specifc_query_synthesizer
1,What is Claude in the context of AI usage?,[Table 1: ChatGPT daily message counts (millio...,Claude is mentioned in the context of ChatGPT ...,single_hop_specifc_query_synthesizer
2,How does the variation in ChatGPT usage among ...,[Variation by Occupation Figure 23 presents va...,The provided context reports that 57% of users...,single_hop_specifc_query_synthesizer
3,How does management influence the responsible ...,[Conclusion This paper studies the rapid growt...,The context does not provide specific informat...,single_hop_specifc_query_synthesizer
4,Hw can AI impact on productivity and outside w...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context shows that AI usage, particularly ...",multi_hop_abstract_query_synthesizer
5,How do privacy considerations in data reportin...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that due to privacy-pres...,multi_hop_abstract_query_synthesizer
6,How does the privacy-preserving methodology us...,[<1-hop>\n\nConclusion This paper studies the ...,The privacy-preserving methodology introduced ...,multi_hop_abstract_query_synthesizer
7,H0w does the privacy and data security in AI a...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context explains that ChatGPT's rapid adop...,multi_hop_abstract_query_synthesizer
8,US ChatGPT use mainly in US but also in US how...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The context indicates that ChatGPT usage in th...,multi_hop_specific_query_synthesizer
9,US ChatGPT usage how much non work messages gr...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"In the US, about 70% of ChatGPT consumer queri...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [ ]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [ ]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [20]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [23]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [25]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [28]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in various ways including performing workplace tasks by augmenting or automating human labor, producing writing, software code, spreadsheets, and other digital products. Users engage with AI for information seeking and advice, as well as for creating digital outputs which distinguishes generative AI from traditional web search technologies. Additionally, AI is being used both at work and outside of work, categorized by user intent into Asking (seeking information or advice), Doing (producing output), and Expressing (such as self-expression or role play).'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [29]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [30]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `dopeness_evaluator`:

## LangSmith Evaluation

In [31]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'puzzled-toy-7' at:
https://smith.langchain.com/o/68718161-2494-5261-8742-6d8e3c15787a/datasets/b80c1ce6-ad28-41ad-bd09-9b6b1775827e/compare?selectedSessions=67f3f7d6-c911-4e41-a1dc-4ba1e5b15db0




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Considering the rapid growth of ChatGPT usage ...,The context indicates that while much economic...,None,"The context indicates that as of July 2025, ov...",1,1,0,5.061494,3ee76a6b-35d1-4b79-bdfb-843b0a39c4be,8c934bde-5602-4996-99a8-556d9b190744
1,US ChatGPT use mainly for info and work but al...,The data shows that there has been steady grow...,None,The data shows that about 70% of ChatGPT queri...,1,1,0,2.585838,dc8acec1-0915-4dec-8921-9a46a5661329,d37f9326-2ecd-447d-be22-b4de18d0f0a2
2,How do Handa et al. describe the growth and us...,I don't know.,None,Handa et al. (2025) analyze the rapid growth o...,0,0,0,0.863154,be2573c3-dcf4-4aa3-94bc-e803e462c0ac,db5bbaab-fbb4-4bc0-9682-502139fbc6bf
3,"How do the usage patterns of ChatGPT, as discu...",Based on the provided context from the documen...,None,"According to Handa et al., the growth in ChatG...",1,1,0,4.267649,8301ad6a-fff1-4e49-8d9f-34aa68dca964,9dbd8144-27d5-4f68-ba99-af7b56863789
4,how work and non work use chatgpt impact socie...,"Based on the provided context, both work and n...",None,ChatGPT launched in November 2022 and by July ...,1,1,0,4.715440,7f0ad74c-91f4-4fb7-86c6-fda4c38cd7ac,ac4471d1-7d56-491d-9695-797540b609fd
5,How do the large language models (LLMs) like C...,Based on the context provided:\n\nLarge Langua...,None,"ChatGPT, launched in November 2022 and based o...",1,1,0,4.654823,84d4f0a6-0869-42c9-a159-81adc948f63c,f8038411-0021-4539-b980-be220372f3d7
6,how ChatGPT use for work and non work and impa...,Based on the provided context:\n\n- **ChatGPT ...,None,ChatGPT launched in November 2022 and by July ...,1,1,0,5.846413,06e3b6c1-ff99-4953-ac5e-04a091f73cbb,11201e4d-e42c-4840-8d11-cccc3019d60f
7,how chatgpt use in work activities and tasks r...,Based on the provided context:\n\n**ChatGPT us...,None,the context shows that variation in chatgpt us...,1,1,0,9.109327,b0a8704d-8feb-4f7f-95ce-93e7e82f1282,a53c855e-b6a1-41aa-b722-616d4dde8819
8,Who is Brynjolfsson in relation to this study?,"Based on the provided context, Brynjolfsson ap...",None,The context references Collis and Brynjolfsson...,0,0,0,1.634198,009dd34f-034a-4836-9763-9f1f57599c20,ab05705f-7f08-4036-8062-dab34380ed6e
9,How is ChatGPT utilized across different occup...,According to the variation presented in Figure...,None,Variation by occupation Figure 23 presents var...,1,1,0,3.196235,e28a55be-312c-47c2-a022-b22495280333,a84e5265-bfd6-49d6-8045-c64b261f325a


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [32]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [33]:
rag_documents = docs

In [34]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    # chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

In [36]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, let’s crank this up to eleven and break down the money moves with AI from the context you dropped:\n\nPeople aren’t just treating ChatGPT like some soulless robot punching tasks out—they’re using it as their slick advisor and research sidekick. This AI hustle isn’t just about automating grunt work; it’s about supercharging decision-making, especially in brain-busting, knowledge-heavy gigs. So imagine a knowledge pro leveling up their output because their AI buddy is whispering genius insights and strategic nudges.\n\nOn top of that, the economic flex is massive. According to Collis and Brynjolfsson (2025), the US alone would need to shell out $98 just to get folks to stop using generative AI for a month—which blows out to a whopping $97 billion annual consumer surplus. Translation? AI isn’t just boosting salaries; it’s creating insane value by grinding smarter, not harder.\n\nSo money-wise: folks are cashing in by riding AI’s decision support wave, making smarter calls faster

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'pertinent-hope-32' at:
https://smith.langchain.com/o/68718161-2494-5261-8742-6d8e3c15787a/datasets/b80c1ce6-ad28-41ad-bd09-9b6b1775827e/compare?selectedSessions=713e7639-4134-414a-88fc-7a181d58a64b




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Considering the rapid growth of ChatGPT usage ...,"Alright, strap in for some next-level insight—...",None,"The context indicates that as of July 2025, ov...",1,1,1,7.384243,3ee76a6b-35d1-4b79-bdfb-843b0a39c4be,aa86e01e-c90d-49c1-8671-e20a121b3c04
1,US ChatGPT use mainly for info and work but al...,"Yo, here’s the lowdown straight from the digit...",None,The data shows that about 70% of ChatGPT queri...,0,0,1,6.650600,dc8acec1-0915-4dec-8921-9a46a5661329,abbbdbec-8cff-4dfc-ac5f-cda81b194a52
2,How do Handa et al. describe the growth and us...,"Yo, here’s the scoop straight from the ChatGPT...",None,Handa et al. (2025) analyze the rapid growth o...,0,0,1,2.280009,be2573c3-dcf4-4aa3-94bc-e803e462c0ac,4bef9746-14b7-4b61-9b8e-25aa12bd9e25
3,"How do the usage patterns of ChatGPT, as discu...","Alright, here’s the straight-up dopest scoop o...",None,"According to Handa et al., the growth in ChatG...",1,1,1,8.584929,8301ad6a-fff1-4e49-8d9f-34aa68dca964,a017d698-5532-4575-a253-52265017c1d9
4,how work and non work use chatgpt impact socie...,"Alright, let’s crank this up to eleven on the ...",None,ChatGPT launched in November 2022 and by July ...,1,1,1,11.855915,7f0ad74c-91f4-4fb7-86c6-fda4c38cd7ac,d5037a1b-fd7a-4f4b-bc71-e403d6805266
5,How do the large language models (LLMs) like C...,"Alright, buckle up—here’s the next-level scoop...",None,"ChatGPT, launched in November 2022 and based o...",1,1,1,7.313163,84d4f0a6-0869-42c9-a159-81adc948f63c,30937e0c-65d2-4d76-871b-ade6d74230ee
6,how ChatGPT use for work and non work and impa...,"Alright, buckle up for the turbocharged scoop ...",None,ChatGPT launched in November 2022 and by July ...,1,1,1,11.215325,06e3b6c1-ff99-4953-ac5e-04a091f73cbb,e465b5df-2cd9-40b4-b06b-a32f70d1cd88
7,how chatgpt use in work activities and tasks r...,"Alright, buckle up for a turbocharged breakdow...",None,the context shows that variation in chatgpt us...,1,1,1,11.260834,b0a8704d-8feb-4f7f-95ce-93e7e82f1282,7dc79ded-2341-408f-b780-204f0dd2a806
8,Who is Brynjolfsson in relation to this study?,"Oh, buckle up, because Brynjolfsson ain't just...",None,The context references Collis and Brynjolfsson...,1,1,1,3.364398,009dd34f-034a-4836-9763-9f1f57599c20,ad5548a2-0bbb-4b4c-8c17-974c96ae46d4
9,How is ChatGPT utilized across different occup...,"Yo, let's dive into the wild ride of how ChatG...",None,Variation by occupation Figure 23 presents var...,1,0,1,8.750981,e28a55be-312c-47c2-a022-b22495280333,8441e0c4-ccd7-4036-ad34-78cdf902baa4


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.